## 프롬프트 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.25 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성
- `2026.03.25` : 2-Phase 그래프 구조 반영, system v2.0 변수 업데이트, scoring에 resource_estimate 주입, risk_analysis에 resource_estimate/risk_interdependencies 추가, similar_project relevance 필터링 반영, final_verdict에 risk_interdependencies/company_context 추가, 가중치 변경(납기 0.10, 전략 0.15), 서버사이드 재계산 검증 추가

In [ ]:
import sys

sys.path.insert(0, "../../")

from dotenv import load_dotenv

load_dotenv()

### 1. 프롬프트 로더 확인

In [ ]:
from backend.app.agent.prompt_loader import load_prompt, load_system_prompt

# 전체 프롬프트 YAML 목록 로드 확인
prompt_names = ["system", "deal_structuring", "scoring", "resource_estimation", "risk_analysis", "similar_project", "final_verdict"]

for name in prompt_names:
    tpl = load_prompt(name)
    print(f"[{name}] version={tpl.version}, has_output_schema={tpl.output_schema is not None}")

In [ ]:
# 프롬프트 구조 상세 확인 (deal_structuring 예시)
tpl = load_prompt("deal_structuring")
print("=== System Prompt (raw template) ===")
print(tpl._system_prompt[:200], "...")
print("\n=== User Prompt (raw template) ===")
print(tpl._user_prompt[:300], "...")
print("\n=== Output Schema ===")
print(tpl.output_schema)

### 2. system.yaml 렌더링

In [ ]:
# system.yaml v2.0에 주입할 변수 준비 — company_context, deal_criteria만 사용
sample_company_context = """[사업 방향] Vision AI/ML 기반 B2B 솔루션 개발 전문기업. 핵심 제품: MLOps 플랫폼, Video Analytics(VA), Vision Foundation Model
[단기 전략] 제품 납품형 사업 우선, 커스터마이징 최소화. 성공률 80% 이상, 추가 공수 3MM 이내. 기존 고객 재계약 및 확장 우선. SI 지양."""

sample_deal_criteria = """최소 마진율: 10% 이상 (전략적 예외 허용)
선호 산업군: 제조업, 건설, 플랜트(정유/화학/에너지), 조선, 공공
최소 계약 규모: 8,000만원 이상
수행 기간: 2~12개월
기술 선호: 클라우드(AWS/GCP) 우선
자사 제품(MLOps/VA) 활용 가능 프로젝트 우선, 비Vision AI/순수 SI 제외"""

# 현재 가중치 기준 (납기 리스크 0.10, 전략적 가치 0.15)
sample_scoring_criteria = [
    {"name": "기술 적합성", "weight": 0.20, "description": "자사 기술 스택(Vision AI, ML, Computer Vision)과의 적합도"},
    {"name": "수익성", "weight": 0.20, "description": "예상 수익 대비 비용 효율. 마진율 20% 이상 목표"},
    {"name": "리소스 가용성", "weight": 0.15, "description": "현재 투입 가능 인력 수준 및 핵심 역할 확보"},
    {"name": "납기 리스크", "weight": 0.10, "description": "일정 준수 가능성, 버퍼 적절성"},
    {"name": "요구사항 명확성", "weight": 0.10, "description": "요구사항의 명확도 및 범위 변경 가능성"},
    {"name": "전략적 가치", "weight": 0.15, "description": "레퍼런스 케이스, 신규 시장 진출, 기술 축적, 장기 파트너십"},
    {"name": "고객 리스크", "weight": 0.10, "description": "고객사 안정성, 지불 능력, 의사결정 명확성"},
]
print(f"가중치 합계: {sum(c['weight'] for c in sample_scoring_criteria)}")

In [ ]:
# system.yaml v2.0 렌더링 — company_context, deal_criteria 2개 변수만 사용
system_tpl = load_system_prompt()
system_base = system_tpl.render_system(
    company_context=sample_company_context,
    deal_criteria=sample_deal_criteria,
)
print(system_base)

### 3. deal_structuring 프롬프트 → LLM 호출

In [ ]:
from backend.app.agent.base import call_llm, parse_json_response

# 샘플 딜 입력 텍스트 — [딜 기본 정보] + [상세 내용] 이중 섹션 형식
sample_deal_input = """[딜 기본 정보]
고객사: 메가테크
고객 규모: 대기업
산업군: 제조업
프로젝트명: 공장 설비 예지보전 AI 시스템 개발
예상 금액: 20000만원
금액 단위: 만원
기간: 5개월

[상세 내용]
## 프로젝트 배경
기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감을 목표로 함.
현재 설비 고장으로 인한 비계획 정지가 연간 약 50회 발생하며, 이를 80% 이상 감소시키는 것이 핵심 목표.

## 기술 요구사항
- Python, TensorFlow 기반 시계열 예측 모델
- IoT 센서 데이터 실시간 수집/처리 파이프라인
- 대시보드 (React 기반)
- MLOps: 모델 재학습 자동화 파이프라인

## 데이터 요구사항
- 3년간 설비 센서 데이터 (진동, 온도, 전류) 약 500GB
- 설비 고장 이력 데이터

## 보안 요구사항
- 공장 내 보안 네트워크 환경에서만 운영
- 온프레미스 배포 필수
- 외부 클라우드 연결 불가

## 결제 조건
착수금 30%, 중간 40%, 완료 30%

## 이해관계자
- 발주측 PM: 설비관리팀 김부장
- 의사결정권자: 제조본부장
- IT 지원: 자체 IT 인프라팀 보유"""

In [ ]:
# deal_structuring 프롬프트 렌더링
tpl = load_prompt("deal_structuring")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    deal_input=sample_deal_input,
)

print("=== System Prompt ===")
print(system_prompt[:500], "...")
print("\n=== User Prompt ===")
print(user_prompt)

In [ ]:
# LLM 호출 및 파싱
raw_output = await call_llm(system_prompt, user_prompt)
print("=== LLM Raw Output ===")
print(raw_output)

In [ ]:
structured_deal = parse_json_response(raw_output)

import json

print("=== Parsed Structured Deal ===")
print(json.dumps(structured_deal, ensure_ascii=False, indent=2))

# 21개 필드 존재 확인
expected_fields = {
    "customer_name", "customer_size", "customer_industry", "decision_context",
    "project_type", "project_overview",
    "tech_requirements", "tech_complexity_notes", "data_requirements", "integration_systems",
    "expected_amount", "amount_unit", "duration_months", "key_milestones", "payment_terms",
    "security_requirements", "constraints", "stakeholders",
    "missing_fields",
}
actual_fields = set(structured_deal.keys())
missing_from_output = expected_fields - actual_fields
extra_fields = actual_fields - expected_fields
print(f"\n=== 필드 검증 ===")
print(f"기대 필드 중 누락: {missing_from_output or '없음'}")
print(f"추가 필드: {extra_fields or '없음'}")

# missing_fields 검증 — 6개 코어 필드만 허용
core_fields = {"customer_name", "customer_industry", "project_overview", "tech_requirements", "expected_amount", "duration_months"}
actual_missing = set(structured_deal.get("missing_fields", []))
invalid_missing = actual_missing - core_fields
print(f"missing_fields: {structured_deal.get('missing_fields', [])}")
if invalid_missing:
    print(f"WARNING: 코어 필드가 아닌 항목이 missing_fields에 포함됨: {invalid_missing}")
print(f"amount_unit: {structured_deal.get('amount_unit', 'N/A')}")

### 4. resource_estimation 프롬프트 → LLM 호출 (Phase 1)

현재 그래프에서 resource_estimation은 **Phase 1**에서 실행되며, scoring/risk_analysis보다 먼저 완료됩니다.  
scoring과 risk_analysis가 resource_estimate 결과를 참조하므로 이 순서를 따릅니다.

In [ ]:
# scoring 프롬프트 렌더링 셀은 아래에서 재구성
# 먼저 resource_estimation 프롬프트를 실행

# 팀 멤버 데이터 — 기본 10명 (configs/defaults/team_members.yaml 기준)
sample_team_members = [
    {"name": "김BE", "role": "BE", "monthly_rate": 1500, "is_available": True, "available_from": None},
    {"name": "이BE", "role": "BE", "monthly_rate": 2000, "is_available": False, "available_from": "2026-05-01"},
    {"name": "김dev", "role": "DevOps", "monthly_rate": 1000, "is_available": True, "available_from": None},
    {"name": "이dev", "role": "DevOps", "monthly_rate": 1500, "is_available": True, "available_from": None},
    {"name": "김FE", "role": "FE", "monthly_rate": 1500, "is_available": True, "available_from": None},
    {"name": "이FE", "role": "FE", "monthly_rate": 1000, "is_available": False, "available_from": "2026-06-01"},
    {"name": "김ML", "role": "MLE", "monthly_rate": 2000, "is_available": True, "available_from": None},
    {"name": "이ML", "role": "MLE", "monthly_rate": 1500, "is_available": True, "available_from": None},
    {"name": "김PM", "role": "PM", "monthly_rate": 2000, "is_available": True, "available_from": None},
    {"name": "이PM", "role": "PM", "monthly_rate": 1000, "is_available": False, "available_from": "2026-04-15"},
]

sample_company_rates = "BE: 월 1,500~2,000만원, DevOps: 월 1,000~1,500만원, FE: 월 1,000~1,500만원, MLE: 월 1,500~2,000만원, PM: 월 1,000~2,000만원"

# 유사 프로젝트 테스트 데이터 (Pinecone 검색 결과 시뮬레이션)
sample_past_projects = [
    {
        "project_name": "S전자 반도체 설비 이상 탐지 시스템",
        "similarity_score": 0.82,
        "industry": "제조업",
        "tech_stack": ["Python", "TensorFlow", "Kafka", "React"],
        "duration_months": 6,
        "planned_headcount": 5,
        "actual_headcount": 6,
        "contract_amount": 25000,
        "summary": "반도체 공정 설비 센서 데이터 기반 이상 탐지 및 예지보전 시스템. 실시간 스트리밍 처리 포함.",
    },
]

In [ ]:
tpl = load_prompt("resource_estimation")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    team_members=sample_team_members,
    company_rates=sample_company_rates,
    past_projects=sample_past_projects,
    company_context=sample_company_context,
)

print("=== User Prompt (resource_estimation) ===")
print(user_prompt[:1000], "...")

### resource_estimation 출력 및 스키마 검증

In [ ]:
raw_output = await call_llm(system_prompt, user_prompt)
resource_result = parse_json_response(raw_output)

print("=== Resource Estimation Result ===")
print(json.dumps(resource_result, ensure_ascii=False, indent=2))

# 출력 스키마 검증
expected_keys = {"work_breakdown", "phases", "team_composition", "duration_months",
                 "risk_buffer_ratio", "duration_with_buffer", "cost_breakdown", "profitability", "rationale"}
actual_keys = set(resource_result.keys())
print(f"\n=== 스키마 검증 ===")
print(f"기대 키 중 누락: {expected_keys - actual_keys or '없음'}")

# profitability 서브키 검증
if "profitability" in resource_result:
    prof = resource_result["profitability"]
    print(f"expected_margin: {prof.get('expected_margin')} (범위: 0.0~1.0)")
    print(f"margin_assessment: {prof.get('margin_assessment')}")

# duration_with_buffer 계산 검증
dm = resource_result.get("duration_months", 0)
rbr = resource_result.get("risk_buffer_ratio", 0)
dwb = resource_result.get("duration_with_buffer", 0)
expected_dwb = round(dm * (1 + rbr), 1)
print(f"duration_with_buffer 검증: {dm} * (1 + {rbr}) = {expected_dwb} (실제: {dwb})")

### 5. similar_project 프롬프트 → LLM 호출 (Phase 1)

similar_project도 Phase 1에서 resource_estimation과 병렬 실행됩니다.  
relevance_score < 0.4 프로젝트는 excluded_projects로 분류됩니다.

In [ ]:
tpl = load_prompt("similar_project")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    past_projects=sample_past_projects,
)

raw_output = await call_llm(system_prompt, user_prompt)
similar_result = parse_json_response(raw_output)

print("=== Similar Project Result ===")
print(json.dumps(similar_result, ensure_ascii=False, indent=2))

# relevance 필터링 검증
for p in similar_result.get("similar_projects", []):
    assert p.get("relevance_score", 0) >= 0.4, f"relevance_score < 0.4 프로젝트가 포함됨: {p['project_name']}"
    print(f"  ✓ {p['project_name']}: relevance={p.get('relevance_score')}, similarity={p.get('similarity_score')}")

for p in similar_result.get("excluded_projects", []):
    print(f"  ✗ (제외) {p['project_name']}: {p.get('excluded_reason')}")

### 6. scoring 프롬프트 → LLM 호출 (Phase 2)

Phase 2에서 실행. resource_estimate 결과를 참조하여 수익성/리소스/납기 평가 정확도를 높입니다.  
**서버사이드 재계산**: LLM이 반환한 score와 weight로 weighted_score를 서버에서 재계산합니다.

In [ ]:
# scoring 프롬프트 — resource_estimate 주입
tpl = load_prompt("scoring")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    scoring_criteria=sample_scoring_criteria,
    resource_estimate=resource_result,
    company_context=sample_company_context,
)

raw_output = await call_llm(system_prompt, user_prompt)
scoring_result = parse_json_response(raw_output)

print("=== LLM Scoring Result (raw) ===")
print(json.dumps(scoring_result, ensure_ascii=False, indent=2))

# 서버사이드 재계산 시뮬레이션
scores = scoring_result.get("scores", [])
recalculated = []
total_score = 0.0
for s in scores:
    score = float(s.get("score", 0))
    weight = float(s.get("weight", 0))
    weighted = round(score * weight, 2)
    total_score += weighted
    recalculated.append({
        "criterion": s.get("criterion", ""),
        "score": score,
        "weight": weight,
        "weighted_score": weighted,
        "rationale": s.get("rationale", ""),
    })

total_score = round(total_score, 2)

# verdict 결정 (서버 로직)
if total_score >= 70:
    verdict = "go"
elif total_score >= 40:
    verdict = "conditional_go"
else:
    verdict = "no_go"

print(f"\n=== 서버사이드 재계산 결과 ===")
for s in recalculated:
    print(f"  {s['criterion']}: {s['score']}점 × {s['weight']} = {s['weighted_score']} — {s['rationale'][:60]}...")
print(f"\n  총점: {total_score} → {verdict}")

### 7. risk_analysis 프롬프트 → LLM 호출 (Phase 2)

Phase 2에서 scoring과 병렬 실행. resource_estimate를 참조하여 재무/일정 리스크를 구체화합니다.  
**신규**: `risk_interdependencies` (리스크 간 상호작용) 출력을 검증합니다.

In [ ]:
# risk_analysis 프롬프트 — resource_estimate 주입
tpl = load_prompt("risk_analysis")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    resource_estimate=resource_result,
    company_context=sample_company_context,
)

raw_output = await call_llm(system_prompt, user_prompt)
risk_result = parse_json_response(raw_output)

print("=== Risk Analysis Result ===")
print(json.dumps(risk_result, ensure_ascii=False, indent=2))

# risks 필드 검증
valid_categories = {"기술", "일정", "재무", "고객", "범위"}
valid_levels = {"HIGH", "MEDIUM", "LOW"}
for r in risk_result.get("risks", []):
    cat = r.get("category", "")
    lvl = r.get("level", "")
    assert cat in valid_categories, f"잘못된 category: {cat}"
    assert lvl in valid_levels, f"잘못된 level: {lvl}"
    print(f"  [{lvl}] {cat} — {r.get('item', '')}: prob={r.get('probability')}, impact={r.get('impact')}")

# risk_interdependencies 검증 (신규)
interdeps = risk_result.get("risk_interdependencies", [])
print(f"\n=== Risk Interdependencies ({len(interdeps)}건) ===")
for ri in interdeps:
    print(f"  {ri.get('risk_pair', [])} → {ri.get('combined_effect', '')}")

### 8. final_verdict 프롬프트 → 마크다운 리포트

모든 이전 노드 결과를 종합하여 8개 섹션 마크다운 리포트를 생성합니다.  
**변경**: `risk_interdependencies`, `company_context` 변수 추가.

In [ ]:
tpl = load_prompt("final_verdict")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    scores=recalculated,
    total_score=total_score,
    verdict=verdict,
    resource_estimate=resource_result,
    risks=risk_result.get("risks", []),
    risk_interdependencies=risk_result.get("risk_interdependencies", []),
    similar_projects=similar_result.get("similar_projects", []),
    company_context=sample_company_context,
)

print("=== User Prompt (final_verdict) ===")
print(user_prompt[:1500], "...")

In [ ]:
final_report = await call_llm(system_prompt, user_prompt)

print(f"=== Final Verdict Report ===")
print(f"리포트 길이: {len(final_report)} chars")

# 8개 섹션 존재 확인
expected_sections = ["핵심 결론", "Deal 개요", "Deal 선정 기준 적합성", "평가 상세",
                     "예상 작업 범위", "리스크 분석", "유사 프로젝트", "권고사항"]
for section in expected_sections:
    if section in final_report:
        print(f"  ✓ '{section}' 섹션 존재")
    else:
        print(f"  ✗ '{section}' 섹션 누락")

In [ ]:
# 마크다운 렌더링 확인
from IPython.display import Markdown, display

display(Markdown(final_report))